<a href="https://colab.research.google.com/github/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abubakarshahid439/ABUBAKAR.flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We build a feature vector using available search and content signals.
The goal is to transform raw data into numerical features that a model can use.

We include:

Numeric features (search_volume, competition, cpc)
Text-based features (query length)
Engineered features (CTR proxy, keyword length)

Missing values are handled to avoid training errors.

In [1]:
import pandas as pd
import numpy as np

# Sample dataset (replace with your real dataset if available)
data = {
    'query': ['shoes', 'running shoes', 'phone', 'best phone', 'laptop'],
    'search_volume': [1000, 800, 1500, 1200, 900],
    'competition': [0.8, 0.6, 0.9, 0.7, 0.5],
    'cpc': [2.5, 2.0, 3.0, 2.8, 1.5],
    'clicked': [1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

# --- Feature Engineering ---
df['query_length'] = df['query'].apply(len)
df['word_count'] = df['query'].apply(lambda x: len(x.split()))

# CTR proxy (simple example)
df['ctr_proxy'] = df['clicked'] / 1  # assuming 1 impression per row

# Handle missing values
df.fillna(0, inplace=True)

# Final feature vector
features = df[['search_volume', 'competition', 'cpc', 'query_length', 'word_count', 'ctr_proxy']]

print("Feature Vector Shape:", features.shape)
print(features.head())

Feature Vector Shape: (5, 6)
   search_volume  competition  cpc  query_length  word_count  ctr_proxy
0           1000          0.8  2.5             5           1        1.0
1            800          0.6  2.0            13           2        0.0
2           1500          0.9  3.0             5           1        1.0
3           1200          0.7  2.8            10           2        0.0
4            900          0.5  1.5             6           1        1.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

search_volume

Meaning: Number of searches for a keyword
Missing: Filled with 0
Available when predicting: ✅ Yes

competition

Meaning: Level of competition (0 to 1)
Missing: Filled with mean/0
Available: ✅ Yes

cpc (cost per click)

Meaning: Advertiser bid value
Missing: Filled with 0
Available: ✅ Yes

query_length

Meaning: Number of characters in query
Missing: Not applicable
Available: ✅ Yes

word_count

Meaning: Number of words in query
Missing: Not applicable
Available: ✅ Yes

ctr_proxy

Meaning: Simulated click-through rate
Missing: Filled with 0
Available: ❌ No (uses clicked → not available at prediction time)

In [2]:
print("Missing values:\n", df.isnull().sum())
print("\nFeature summary:\n", features.describe())


Missing values:
 query            0
search_volume    0
competition      0
cpc              0
clicked          0
query_length     0
word_count       0
ctr_proxy        0
dtype: int64

Feature summary:
        search_volume  competition       cpc  query_length  word_count  \
count       5.000000     5.000000  5.000000      5.000000    5.000000   
mean     1080.000000     0.700000  2.360000      7.800000    1.400000   
std       277.488739     0.158114  0.610737      3.563706    0.547723   
min       800.000000     0.500000  1.500000      5.000000    1.000000   
25%       900.000000     0.600000  2.000000      5.000000    1.000000   
50%      1000.000000     0.700000  2.500000      6.000000    1.000000   
75%      1200.000000     0.800000  2.800000     10.000000    2.000000   
max      1500.000000     0.900000  3.000000     13.000000    2.000000   

       ctr_proxy  
count   5.000000  
mean    0.600000  
std     0.547723  
min     0.000000  
25%     0.000000  
50%     1.000000  
75%     

## 3. The leakage hunt

We check for leakage by identifying features that use future or label information.

Potential leakage:

clicked (target variable)
ctr_proxy (derived from clicked)

These should not be used for prediction because they contain information that would not be available at prediction time.

In [3]:
# Check correlation with label
correlation = df.corr(numeric_only=True)['clicked'].sort_values(ascending=False)

print("Correlation with label:\n", correlation)

# Identify suspicious features
leakage_features = ['clicked', 'ctr_proxy']
print("\nPotential leakage features:", leakage_features)

Correlation with label:
 clicked          1.000000
ctr_proxy        1.000000
competition      0.288675
search_volume    0.263181
cpc             -0.059788
query_length    -0.947784
word_count      -1.000000
Name: clicked, dtype: float64

Potential leakage features: ['clicked', 'ctr_proxy']


## 4. What I excluded and why

-clicked

Reason: This is the target variable

-ctr_proxy

Reason: Derived from clicked → causes leakage

-query (raw text)

Reason: Not directly usable without encoding

In [4]:
# Final safe features (no leakage)
final_features = df[['search_volume', 'competition', 'cpc', 'query_length', 'word_count']]

print("Final Features Used:\n", final_features.head())

Final Features Used:
    search_volume  competition  cpc  query_length  word_count
0           1000          0.8  2.5             5           1
1            800          0.6  2.0            13           2
2           1500          0.9  3.0             5           1
3           1200          0.7  2.8            10           2
4            900          0.5  1.5             6           1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.